<a href="https://colab.research.google.com/github/asopozala-prog/ecommerce-ml-experiments-bqml/blob/main/imdb_sentiment_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMDB Sentiment Analysis in Google Colab

This notebook builds a real end-to-end sentiment analysis model on the IMDB movie reviews dataset using TensorFlow and Keras. It demonstrates how to:

- Use `tf.data` pipelines with raw text
- Tokenize text with `TextVectorization`
- Learn word representations with an `Embedding` layer
- Build a bidirectional LSTM classifier for positive/negative sentiment
- Evaluate the model and run predictions on custom reviews

Everything runs in Google Colab and can take advantage of a free GPU.


In [1]:
import tensorflow as tf
print(tf.__version__)


2.20.0


In [2]:
import tensorflow_datasets as tfds

# Load IMDB with raw text
train_data, test_data = tfds.load(
    "imdb_reviews",
    split=["train", "test"],
    as_supervised=True  # returns (text, label) pairs
)

train_data, test_data


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.DBROYK_1.0.0/imdb_reviews-train.tfrecor…

...

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.DBROYK_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.DBROYK_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


(<_PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>,
 <_PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>)

In [3]:
BUFFER_SIZE = 10000
BATCH_SIZE = 64

# Shuffle and batch train data
train_data = train_data.shuffle(BUFFER_SIZE, reshuffle_each_iteration=True)

val_size = 5000
val_data = train_data.take(val_size).batch(BATCH_SIZE)
train_data = train_data.skip(val_size).batch(BATCH_SIZE)

test_data = test_data.batch(BATCH_SIZE)


In [4]:
from tensorflow.keras.layers import TextVectorization

VOCAB_SIZE = 20000      # max vocabulary size
SEQUENCE_LENGTH = 200   # truncate/pad to this length

# TextVectorization layer
vectorize_layer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH
)

# We need a dataset of just texts to adapt on:
train_text = train_data.map(lambda text, label: text)

# Adapt the layer to the training text
vectorize_layer.adapt(train_text)


In [5]:
def vectorize_text(text, label):
    text = tf.expand_dims(text, -1)  # TextVectorization expects rank-2/3
    return vectorize_layer(text), label

train_ds = train_data.map(vectorize_text)
val_ds = val_data.map(vectorize_text)
test_ds = test_data.map(vectorize_text)


In [6]:
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)


In [7]:
from tensorflow.keras import layers, models

EMBEDDING_DIM = 128

inputs = layers.Input(shape=(SEQUENCE_LENGTH,), dtype="int64")

# 1) Embedding: maps word IDs to dense vectors
x = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBEDDING_DIM,
    name="word_embedding"
)(inputs)

# 2) Encoder: Bidirectional LSTM
x = layers.Bidirectional(
    layers.LSTM(64, return_sequences=False)
)(x)

# 3) Decoder/head: Dense layers for classification
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ word_embedding (Embedding)      │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,667,137 (10.17 MB)

 Trainable params: 2,667,137 (10.17 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 179s 553ms/step - accuracy: 0.7464 - loss: 0.5123 - val_accuracy: 0.7126 - val_loss: 0.5779
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 179s 571ms/step - accuracy: 0.8783 - loss: 0.3082 - val_accuracy: 0.9226 - val_loss: 0.2258
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 172s 543ms/step - accuracy: 0.9262 - loss: 0.2082 - val_accuracy: 0.9370 - val_loss: 0.1815
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 170s 543ms/step - accuracy: 0.9482 - loss: 0.1494 - val_accuracy: 0.9502 - val_loss: 0.1443
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 170s 543ms/step - accuracy: 0.9617 - loss: 0.1178 - val_accuracy: 0.9604 - val_loss: 0.1175


In [9]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy:", test_acc)


391/391 ━━━━━━━━━━━━━━━━━━━━ 61s 156ms/step - accuracy: 0.8128 - loss: 0.4866
Test accuracy: 0.8128399848937988


In [13]:
import numpy as np

def predict_review(text):
    # Wrap text into a batch of size 1
    text_tensor = tf.constant([text])
    # Vectorize
    vectorized = vectorize_layer(text_tensor)
    # Predict
    prob = model.predict(vectorized)[0][0]
    label = "positive" if prob >= 0.5 else "negative"
    return label, float(prob)

samples = [
    "An amiable, warm-hearted, inoffensive family animated film about a dog who stars in a popular television show.",
    "Bolt is certainly not one of Disney's greatest hits.",
]

for s in samples:
    label, p = predict_review(s)
    print(f"Review: {s}\nPrediction: {label} (p={p:.3f})\n")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Review: An amiable, warm-hearted, inoffensive family animated film about a dog who stars in a popular television show.
Prediction: positive (p=0.799)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
Review: Bolt is certainly not one of Disney's greatest hits.
Prediction: positive (p=0.795)



In [15]:
from google.colab import drive
drive.mount('/content/drive')

model.save('/content/drive/MyDrive/imdb_sentiment_model.keras')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
model.save('/content/drive/MyDrive/imdb_sentiment_model.keras')


from tensorflow.keras.models import load_model

model = load_model('/content/drive/MyDrive/imdb_sentiment_model.keras')

